# Lab 06 — 00 Source Preparation

**Lab:** Gold Layer & Business Analytics  
**Dataset:** Synthea Healthcare  
**Storage:** Unity Catalog **EXTERNAL volume**

## Purpose

This notebook prepares the Synthea source data for Lab 06.

It:

1. reads runtime parameters,
2. validates the external volume,
3. prepares source/reference/landing folders,
4. downloads the Synthea 1K CSV archive from the official Synthea GitHub organization,
5. extracts CSV files directly into the external volume,
6. validates the six source datasets used by Lab 06,
7. profiles source row counts,
8. validates required source columns,
9. profiles encounter coverage,
10. stages reference/master data,
11. validates staging,
12. produces a final completion summary.

### Design rules

- The external volume is created by the Databricks Bundle, **not inside this notebook**.
- No `current_user()`-generated paths.
- Environment values are parameterized.
- Retry behavior belongs at Job/task level.
- Reference data is separated from transactional encounter data.
- Encounter monthly batching will be implemented separately in `tools/synthea_batch_loader.py`.

## 1. Runtime parameters

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "01 Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "02 Schema")
dbutils.widgets.text("volume_name", "lab06_gold_analytics", "03 External volume")

dbutils.widgets.text(
    "synthea_url",
    "https://raw.githubusercontent.com/synthetichealth/synthea-sample-data/main/downloads/synthea_sample_data_csv_nov2021.zip",
    "04 Synthea ZIP URL",
)

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
synthea_url = dbutils.widgets.get("synthea_url")

volume_fqn = f"{catalog}.{schema}.{volume_name}"
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

source_path = f"{volume_path}/source"
csv_source_path = f"{source_path}/csv"
reference_path = f"{volume_path}/reference"
encounter_landing_path = f"{volume_path}/landing/encounters"

print(f"Catalog                 : {catalog}")
print(f"Schema                  : {schema}")
print(f"External volume         : {volume_name}")
print(f"Volume FQN              : {volume_fqn}")
print(f"Volume path             : {volume_path}")
print(f"CSV source path         : {csv_source_path}")
print(f"Reference path          : {reference_path}")
print(f"Encounter landing path  : {encounter_landing_path}")

## 2. Validate EXTERNAL volume

The volume must already exist and must be of type `EXTERNAL`.

In [0]:
volume_df = spark.sql(f"DESCRIBE VOLUME {volume_fqn}")
display(volume_df)

volume_row = volume_df.first().asDict()

volume_type = volume_row.get("volume_type")
storage_location = volume_row.get("storage_location")

if volume_type != "EXTERNAL":
    raise ValueError(
        f"{volume_fqn} is {volume_type}; Lab 06 expects EXTERNAL."
    )

if not storage_location:
    raise ValueError(
        f"{volume_fqn} has no storage_location."
    )

print("External volume validation passed.")
print(f"Storage location: {storage_location}")

## 3. Prepare folders inside the external volume

In [0]:
for path in [
    csv_source_path,
    reference_path,
    encounter_landing_path,
]:
    dbutils.fs.mkdirs(path)

print("Lab 06 folders are ready.")

## 4. Download and extract Synthea directly to the external volume

The archive is kept in memory and CSV files are written directly to `/Volumes/...`.

This avoids restricted `file:/tmp → dbutils.fs.cp(...)` operations.

In [0]:
import io
import os
import shutil
import zipfile
import requests

print(f"Downloading: {synthea_url}")

response = requests.get(
    synthea_url,
    timeout=120,
)
response.raise_for_status()

archive_size_mb = len(response.content) / (1024 * 1024)

print(f"Archive downloaded: {archive_size_mb:.2f} MB")

copied_files = []

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    csv_members = [
        name
        for name in archive.namelist()
        if name.startswith("csv/")
        and name.lower().endswith(".csv")
    ]

    if not csv_members:
        raise ValueError(
            "No CSV files were found inside the Synthea archive."
        )

    for member in sorted(csv_members):
        filename = os.path.basename(member)
        target_file = f"{csv_source_path}/{filename}"

        with archive.open(member) as source_handle:
            with open(target_file, "wb") as target_handle:
                shutil.copyfileobj(
                    source_handle,
                    target_handle,
                )

        copied_files.append(filename)

print(f"Copied {len(copied_files)} CSV files directly to:")
print(csv_source_path)

## 5. Verify source files in the external volume

In [0]:
source_files = dbutils.fs.ls(csv_source_path)

display(source_files)

print(f"Source files found: {len(source_files)}")

## 6. Define and validate the six Lab 06 source datasets

The archive contains more healthcare tables, but Lab 06 initially uses:

- `patients.csv`
- `encounters.csv`
- `providers.csv`
- `organizations.csv`
- `payers.csv`
- `conditions.csv`

In [0]:
REQUIRED_FILES = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "providers": "providers.csv",
    "organizations": "organizations.csv",
    "payers": "payers.csv",
    "conditions": "conditions.csv",
}

available_files = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(csv_source_path)
}

missing_files = sorted(
    set(REQUIRED_FILES.values()) - available_files
)

source_file_validation = [
    (
        dataset,
        filename,
        "PASS" if filename in available_files else "FAIL",
    )
    for dataset, filename in REQUIRED_FILES.items()
]

source_file_validation_df = spark.createDataFrame(
    source_file_validation,
    ["dataset", "file_name", "status"],
)

display(
    source_file_validation_df.orderBy("dataset")
)

if missing_files:
    raise FileNotFoundError(
        "Missing required Synthea files: "
        + ", ".join(missing_files)
    )

print("All six required Synthea source files are available.")

## 7. Load and profile source datasets

`inferSchema` is used here for source exploration only.
Gold target types will be controlled explicitly later.

In [0]:
source_dfs = {}
inventory = []

for dataset, filename in REQUIRED_FILES.items():
    path = f"{csv_source_path}/{filename}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    source_dfs[dataset] = df

    row_count = df.count()
    column_count = len(df.columns)

    inventory.append(
        (
            dataset,
            filename,
            row_count,
            column_count,
        )
    )

inventory_df = spark.createDataFrame(
    inventory,
    [
        "dataset",
        "file_name",
        "row_count",
        "column_count",
    ],
)

display(
    inventory_df.orderBy("dataset")
)

## 8. Validate required source columns

These checks verify the minimum columns required for the planned Gold model.

In [0]:
REQUIRED_COLUMNS = {
    "patients": {
        "Id",
    },
    "encounters": {
        "Id",
        "START",
        "STOP",
        "PATIENT",
        "ORGANIZATION",
        "PROVIDER",
        "PAYER",
    },
    "providers": {
        "Id",
    },
    "organizations": {
        "Id",
    },
    "payers": {
        "Id",
    },
    "conditions": {
        "PATIENT",
        "CODE",
        "DESCRIPTION",
    },
}

validation_rows = []
failures = []

for dataset, required_columns in REQUIRED_COLUMNS.items():
    actual_columns = set(
        source_dfs[dataset].columns
    )

    missing_columns = sorted(
        required_columns - actual_columns
    )

    status = (
        "PASS"
        if not missing_columns
        else "FAIL"
    )

    validation_rows.append(
        (
            dataset,
            status,
            ", ".join(missing_columns),
        )
    )

    if missing_columns:
        failures.append(
            f"{dataset}: missing "
            + ", ".join(missing_columns)
        )

column_validation_df = spark.createDataFrame(
    validation_rows,
    [
        "dataset",
        "status",
        "missing_columns",
    ],
)

display(
    column_validation_df.orderBy("dataset")
)

if failures:
    raise ValueError(
        "Source-column validation failed:\n"
        + "\n".join(failures)
    )

print("Required source-column validation passed.")

## 9. Profile encounter coverage

This gives the date range and core entity counts required for:

- `dim_date`,
- monthly encounter batches,
- fact modeling,
- dashboard planning,
- alert-volume simulation.

In [0]:
from pyspark.sql import functions as F

encounter_profile_df = (
    source_dfs["encounters"]
    .select(
        F.min(
            F.to_timestamp("START")
        ).alias("min_encounter_start"),

        F.max(
            F.to_timestamp("START")
        ).alias("max_encounter_start"),

        F.count("*").alias(
            "encounter_rows"
        ),

        F.countDistinct(
            "PATIENT"
        ).alias(
            "unique_patients"
        ),

        F.countDistinct(
            "ORGANIZATION"
        ).alias(
            "unique_organizations"
        ),

        F.countDistinct(
            "PROVIDER"
        ).alias(
            "unique_providers"
        ),

        F.countDistinct(
            "PAYER"
        ).alias(
            "unique_payers"
        ),
    )
)

display(encounter_profile_df)

## 10. Profile encounter volume by month

This profile will later help us choose a realistic month for the controlled volume-drop alert simulation.

In [0]:
monthly_encounter_profile_df = (
    source_dfs["encounters"]
    .withColumn(
        "encounter_start",
        F.to_timestamp("START"),
    )
    .withColumn(
        "encounter_month",
        F.date_format(
            "encounter_start",
            "yyyy-MM",
        ),
    )
    .groupBy(
        "encounter_month"
    )
    .agg(
        F.count("*").alias(
            "encounter_count"
        ),
        F.countDistinct(
            "PATIENT"
        ).alias(
            "unique_patients"
        ),
    )
    .orderBy(
        "encounter_month"
    )
)

display(monthly_encounter_profile_df)

## 11. Stage reference/master datasets

These are treated as reference/master data:

- patients
- providers
- organizations
- payers
- conditions

`encounters.csv` remains transactional and will later be split into monthly landing batches.

In [0]:
REFERENCE_DATASETS = {
    "patients",
    "providers",
    "organizations",
    "payers",
    "conditions",
}

for dataset in sorted(REFERENCE_DATASETS):
    filename = REQUIRED_FILES[dataset]

    source_file = (
        f"{csv_source_path}/{filename}"
    )

    target_file = (
        f"{reference_path}/{filename}"
    )

    dbutils.fs.cp(
        source_file,
        target_file,
        True,
    )

    print(
        f"Staged: {filename}"
    )

## 12. Validate staged reference data

In [0]:
expected_reference_files = {
    REQUIRED_FILES[dataset]
    for dataset in REFERENCE_DATASETS
}

staged_reference_files = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(reference_path)
}

reference_validation_rows = [
    (
        filename,
        (
            "PASS"
            if filename in staged_reference_files
            else "FAIL"
        ),
    )
    for filename in sorted(
        expected_reference_files
    )
]

reference_validation_df = spark.createDataFrame(
    reference_validation_rows,
    [
        "file_name",
        "status",
    ],
)

display(reference_validation_df)

missing_reference_files = sorted(
    expected_reference_files
    - staged_reference_files
)

if missing_reference_files:
    raise RuntimeError(
        "Reference staging failed. Missing: "
        + ", ".join(
            missing_reference_files
        )
    )

print(
    "Reference staging validation passed."
)

## 13. Final source-preparation validation

In [0]:
final_checks = [
    (
        "external_volume",
        volume_type == "EXTERNAL",
    ),
    (
        "source_csv_count",
        len(source_files) > 0,
    ),
    (
        "required_files",
        len(missing_files) == 0,
    ),
    (
        "source_columns",
        len(failures) == 0,
    ),
    (
        "reference_staging",
        len(missing_reference_files) == 0,
    ),
]

final_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            (
                "PASS"
                if passed
                else "FAIL"
            ),
        )
        for check_name, passed
        in final_checks
    ],
    [
        "check_name",
        "status",
    ],
)

display(final_validation_df)

failed_checks = [
    name
    for name, passed
    in final_checks
    if not passed
]

if failed_checks:
    raise RuntimeError(
        "Final source-preparation "
        "validation failed: "
        + ", ".join(failed_checks)
    )

## 14. Completion

Expected external-volume layout:

```text
lab06_gold_analytics/
├── source/
│   └── csv/
│       ├── patients.csv
│       ├── encounters.csv
│       ├── providers.csv
│       ├── organizations.csv
│       ├── payers.csv
│       ├── conditions.csv
│       └── ...
│
├── reference/
│   ├── patients.csv
│   ├── providers.csv
│   ├── organizations.csv
│   ├── payers.csv
│   └── conditions.csv
│
└── landing/
    └── encounters/
```

### Next step

Build `tools/synthea_batch_loader.py` to split `encounters.csv` into monthly landing files.

In [0]:
print("LAB 06 — SOURCE PREPARATION COMPLETE")
print("")
print(f"External volume   : {volume_fqn}")
print(f"Source            : {csv_source_path}")
print(f"Reference         : {reference_path}")
print(f"Encounter landing : {encounter_landing_path}")
print("")
print("Next: tools/synthea_batch_loader.py")